# Retrieval benchmark

Compares three systems on the labelled evaluation set:

1. **TF-IDF baseline** — sparse lexical matching.
2. **Sentence-BERT (retrieval only)** — dense vectors, no re-ranking.
3. **Sentence-BERT + re-ranking** — the full two-stage pipeline.

> **The labels are heuristic, not human judgements.** A posting counts as
> relevant when it shares a normalized title family with the query *and* its
> extracted skills overlap by more than 0.4 Jaccard. Those same signals feed the
> retrievers, so these numbers measure internal consistency rather than true
> relevance. They are comparable *between* systems, which is what this notebook
> is for — the absolute values mean very little. `src/build_eval_set.py` lists
> the limitations in full.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, str(Path.cwd().parent))

from src.baseline_tfidf import TFIDFRecommender
from src.build_eval_set import load_eval_set
from src.config import JOBS_PARQUET
from src.evaluate import K_VALUES, run_ablation, run_benchmark
from src.recommender import JobRecommender

TEMPLATE = "plotly_dark"

jobs = pd.read_parquet(JOBS_PARQUET)
eval_set = load_eval_set(jobs=jobs)
sparse = TFIDFRecommender.load()
dense = JobRecommender.load()

meta = eval_set["metadata"]
print(f"{meta['n_queries']} queries over {meta['corpus_size']:,} postings")
print(f"{meta['mean_relevant_per_query']:.1f} relevant postings per query on average")
print(f"drawn from {meta['eligible_queries']:,} eligible queries")

## Run the benchmark

Takes roughly half a minute.

In [ ]:
report = run_benchmark(jobs, eval_set, sparse, dense)

results = pd.DataFrame(report).T
results.round(3)

## NDCG by system

NDCG is the headline number: unlike precision@k it cares *where* in the list a
relevant posting lands, which is what a ranked recommender is actually judged on.

In [ ]:
ndcg = results[[f"NDCG@{k}" for k in K_VALUES]].reset_index(names="system")
long = ndcg.melt(id_vars="system", var_name="metric", value_name="score")

px.bar(
    long,
    x="metric",
    y="score",
    color="system",
    barmode="group",
    template=TEMPLATE,
    title="NDCG by system and cutoff",
    labels={"score": "NDCG", "metric": ""},
    height=460,
)

## Precision and recall against k

Precision falls as k grows — deeper lists dilute the hits. Recall rises, because
a longer list has more chances to contain everything relevant. The gap between
systems is what matters here, not the absolute level.

In [ ]:
curves = []
for system, scores in report.items():
    for k in K_VALUES:
        curves.append({"system": system, "k": k, "metric": "precision", "value": scores[f"P@{k}"]})
        curves.append({"system": system, "k": k, "metric": "recall", "value": scores[f"R@{k}"]})

px.line(
    pd.DataFrame(curves),
    x="k",
    y="value",
    color="system",
    facet_col="metric",
    markers=True,
    template=TEMPLATE,
    title="Precision and recall against cutoff",
    height=420,
)

## Latency

Mean wall-clock time for one query. The embedder is warmed up before timing, so
these exclude the one-off cost of loading the transformer weights.

In [ ]:
latency = results[["latency_ms"]].reset_index(names="system")

px.bar(
    latency,
    x="system",
    y="latency_ms",
    template=TEMPLATE,
    title="Mean query latency",
    labels={"latency_ms": "milliseconds", "system": ""},
    height=420,
)

## Weight ablation

Re-runs the full pipeline with different weights on the three re-ranking
components. The error bars are standard errors over the queries — where they
overlap, the difference between two settings is not evidence of anything.

In [ ]:
ablation = run_ablation(jobs, eval_set, dense)

frame = pd.DataFrame(ablation).T.sort_values("NDCG@10")

figure = go.Figure(
    go.Bar(
        x=frame["NDCG@10"],
        y=frame.index,
        orientation="h",
        error_x={"type": "data", "array": frame["stderr"]},
        marker_color="#D97757",
    )
)
figure.update_layout(
    template=TEMPLATE,
    title="NDCG@10 by re-ranking weights (± standard error)",
    xaxis_title="NDCG@10",
    height=460,
)
figure.show()

frame[["NDCG@10", "stderr", "P@10", "MAP", "wins", "losses"]].round(3)

### Reading the ablation honestly

Dropping either the skills or the semantic component alone costs a lot, so the
hybrid is doing real work. Among the *blends*, though, the standard errors
overlap and the win/loss counts are close to even — on 50 heuristic queries
there is no evidence that one blend genuinely beats another, which is why the
shipped default was left where it is rather than tuned to the top row.